In [1]:
import json
import pandas as pd
import numpy as np
import os
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm as tqdm
from PIL import Image
from diffusers import FluxPipeline
import torch

In [ ]:
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-3.5-medium", torch_dtype=torch.float16)
pipe.to("cuda")

In [2]:
from datasets import load_dataset
from huggingface_hub import login

# Inserisci il tuo token di accesso
import os
from dotenv import load_dotenv
load_dotenv()
access_token = os.environ.get("HF_TOKEN")

# Effettua il login a Hugging Face
login(access_token)

In [ ]:
# Open and read the JSON file
with open('prompt_datasets/df_aimagelab_js.json', 'r') as file:
    data_aimagelab = json.load(file)

# Open and read the JSON file
with open('prompt_datasets/df_CoPro_js.json', 'r') as file:
    data_CoPro = json.load(file)

In [9]:
import gc

In [10]:
# Parametri di generazione
num_inference_steps = 30  # Ridurre i passi di generazione
guidance_scale = 12.5     # Aumentare il guidance scale

In [ ]:
# Definisci la cartella di salvataggio
output_folder = "stable-diffusion-3.5-medium"
os.makedirs(output_folder, exist_ok=True)

In [11]:
# Funzione per generare immagini in batch e aggiornare il DataFrame
def generate_images_and_update_df(df, df_name, batch_size=2):
    """Genera immagini in batch e aggiorna il DataFrame con gli ID delle immagini."""
    
    path = os.path.join(output_folder, df_name)
    os.makedirs(path, exist_ok=True)
    
    image_ids = []
    prompts = df["prompt"].tolist()
    with torch.no_grad():
        # Elaborazione in batch
        for i in tqdm(range(0, len(prompts), batch_size), total=len(prompts)//batch_size, desc=f"Processing {df_name}"):
            batch_prompts = prompts[i:i+batch_size]
            
            try:
                images = pipe(batch_prompts,
                               num_inference_steps=num_inference_steps, 
              guidance_scale=guidance_scale).images  # Genera batch di immagini

                for j, image in enumerate(images):
                    image_id = f"{df_name}_{i+j}.png"
                    image_path = os.path.join(path, image_id)
                    image.save(image_path)
                    image_ids.append(image_id)

                    # Liberare la memoria della GPU dopo ogni batch
                torch.cuda.empty_cache()  # Rimuove dalla memoria la memoria non utilizzata
                gc.collect()  # Forza il garbage collector a liberare memoria
            except Exception as e:
                print(f"Errore nel batch {i}: {e}")
                image_ids.extend([None] * len(batch_prompts))  # Segna errori
            
    df["image_id"] = image_ids
    return df


In [ ]:
# Processa entrambi i DataFrame
df_aimagelab_result = generate_images_and_update_df(df_aimagelab.sample(frac=1), "aimagelab")
df_CoPro_result = generate_images_and_update_df(df_CoPro.sample(frac=1), "CoPro")

# Salva i DataFrame aggiornati
df_aimagelab_result.to_csv(output_folder+"df_aimagelab_updated.csv", index=False)
df_CoPro_result.to_csv(output_folder+"df_CoPro_updated.csv", index=False)

Processing aimagelab:   0%|          | 0/76091 [00:00<?, ?it/s]

c:\Users\DaisLabTSB\.conda\envs\Giando\Lib\site-packages\transformers\models\clip\modeling_clip.py:539: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


  0%|          | 0/30 [00:00<?, ?it/s]